# 03b - Expanding refit, v1 (optional)

This section moved out of notebook 03 v1. It is **optional**.

**What:** in real life, every month we would retrain the model with the new data that just arrived. Here, for each test month t, we refit each model on **all months before t** (with the **same, fixed** hyperparameters from notebook 03 v1) and predict t.

**Why:** the main notebook trains once on 1999-2019 and never updates. This shows if updating the model every month helps. Nothing is tuned here.

⚠️ This notebook uses the test months, so it has its own `RUN_TEST` switch. Run notebook 03 v1 first.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

In [ ]:
# ---------------- CONFIG ----------------
VERSION = "v1"
RUN_TEST = False  # True runs the expanding refit on the test months
SEED = 42
DROP_FEATURES = ["oil_last_chg", "gas_last_chg"]
# ----------------------------------------

np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "processed_data"
print("Data folder:", DATA_DIR)

## 1. Load the data and the fixed hyperparameters

Same table and same features as notebook 03 v1. The hyperparameters come from the JSON file saved there.

In [ ]:
df = pd.read_csv(DATA_DIR / "model_table_macro.csv", index_col="date", parse_dates=True)
df["month"] = df.index.month
test_months = df[df["split"] == "test"].index

feature_table = pd.read_csv(DATA_DIR / "feature_table.csv")
FEATURES = [f for f in feature_table["feature"] if f not in DROP_FEATURES] + ["month"]
NUMERIC_FEATURES = [f for f in FEATURES if f != "month"]

best_params = json.loads((DATA_DIR / f"best_params_macro_{VERSION}.json").read_text())
print(len(FEATURES), "features |", len(test_months), "test months")
print("Random Forest:", best_params["rf"])
print("XGBoost:", best_params["xgb"])

## 2. Rebuild the 3 pipelines

The same pipelines as in notebook 03 v1, with the fixed hyperparameters.

In [ ]:
linear_preprocess = ColumnTransformer(
    [
        ("scale", StandardScaler(), NUMERIC_FEATURES),
        ("month", OneHotEncoder(drop="first"), ["month"]),
    ],
    verbose_feature_names_out=False,
)
linear_pipe = make_pipeline(linear_preprocess, LinearRegression())

rf_pipe = Pipeline([("model", RandomForestRegressor(random_state=SEED))])
rf_pipe.set_params(**best_params["rf"])

# n_jobs=1: on this tiny dataset, using all CPU cores makes XGBoost much slower
xgb_pipe = Pipeline([
    ("model", XGBRegressor(random_state=SEED, objective="reg:squarederror", n_jobs=1)),
])
xgb_pipe.set_params(**best_params["xgb"])

pipelines = {"linear_macro": linear_pipe, "rf_macro": rf_pipe, "xgb_macro": xgb_pipe}
print(list(pipelines))

## 3. Expanding refit

For each test month t: fit on all months before t, predict t.

In [ ]:
if RUN_TEST:
    expanding_predictions = pd.DataFrame(index=test_months)
    expanding_predictions["y"] = df.loc[test_months, "y"]

    for column, pipe in pipelines.items():
        preds = []
        for t in test_months:
            history = df[df.index < t]
            pipe.fit(history[FEATURES], history["y"])
            preds.append(pipe.predict(df.loc[[t], FEATURES])[0])
        expanding_predictions[column] = preds
        print("done:", column)

    display(expanding_predictions.head())
else:
    print("RUN_TEST=False: expanding refit skipped")

## 4. Compare with the fit-once results

The fit-once predictions come from notebook 03 v1, so that notebook must have run with `RUN_TEST=True`.

In [ ]:
if RUN_TEST:
    fit_once = pd.read_csv(DATA_DIR / f"model_predictions_macro_{VERSION}.csv",
                           index_col="month", parse_dates=True)

    rows = []
    for column in pipelines:
        rows.append({
            "model": column,
            "rmse_fit_once": root_mean_squared_error(fit_once["y"], fit_once[column]),
            "rmse_expanding": root_mean_squared_error(expanding_predictions["y"],
                                                      expanding_predictions[column]),
        })

    refit_comparison = pd.DataFrame(rows)
    refit_comparison["version"] = VERSION
    display(refit_comparison.round(4))
else:
    print("RUN_TEST=False: comparison skipped")

## 5. Save

In [ ]:
if RUN_TEST:
    expanding_out = expanding_predictions.rename_axis("month").reset_index()
    expanding_out["version"] = VERSION
    expanding_out.to_csv(DATA_DIR / f"expanding_predictions_macro_{VERSION}.csv", index=False)
    print("Saved the expanding predictions")
else:
    print("RUN_TEST=False: nothing saved")